# RSO-901 Glycol Set Points — Monthly

Chiller glycol **set point** changes over one `month_obs`, from the
`lsst.sal.HVAC.logevent_chillerConfiguration` event. All four HVAC chillers are drawn as **step**
lines on one shared, zoomable panel — a set point is piecewise-constant, so the last value is held to
the end of the window. Genuine changes are also listed in a change-log table.

Known **catastrophic cooling events** (glycol / Dynalene / PCS faults and near-misses, from
`glycol_catastrophic_faults.csv`) are overlaid as vertical **dashed** lines — red for realised faults,
amber for averted near-misses — so set-point behaviour can be read against known incidents.

See `ai_notes.md` in this folder for the topic survey and field reference.

**`month_obs`** is an integer `YYYYMM` (e.g. `202601`). Following the Rubin day-rollover convention
(the observatory day rolls at **UTC−12**, i.e. 12:00 UTC — close to, but not exactly, Chilean local
noon), the window **starts at 12:00 on the first day of the month** and **ends at 11:59 on the last
day of the month**. That excludes the final night of the month; flip `INCLUDE_LAST_NIGHT` below to
extend to the following noon.

In [3]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import calendar

import astropy.units as u
import pandas as pd

from bokeh.io import output_notebook, output_file, save, show
from bokeh.models import ColumnDataSource, HoverTool, Label, Range1d, Span
from bokeh.plotting import figure

# The getDayObs* helpers live in dateTime; importing them from efdUtils is
# deprecated and those shims are removed after w_2026_01.
from lsst.summit.utils.dateTime import getDayObsStartTime
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient

output_notebook()

## Parameters

In [5]:
# Month to analyse, as YYYYMM.
month_obs = 202602

# Include the last night of the month (extend end to the following noon)?
INCLUDE_LAST_NIGHT = False

TOPIC = "lsst.sal.HVAC.logevent_chillerConfiguration"

# Curated CSV of catastrophic glycol / camera-cooling events (faults & near-misses),
# overlaid on the set-point plot as vertical dashed lines.
EVENTS_CSV = "glycol_catastrophic_faults.csv"

# device_id (DeviceId_chiller enum) -> (name, colour).
# Bright, saturated palette so the four step lines stay distinct on a shared panel.
CHILLERS = {
    101: ("chiller01", "#00C2C2"),  # teal
    102: ("chiller02", "#FF7A00"),  # orange
    103: ("chiller03", "#B026D9"),  # purple
    104: ("chiller04", "#1E90FF"),  # blue
}

# Vertical-line colour by event outcome: realised fault vs averted near-miss.
EVENT_COLORS = {"actual": "#E5484D", "near_miss": "#F5A623"}

In [6]:
year, month = divmod(month_obs, 100)
n_days = calendar.monthrange(year, month)[1]

first_day_obs = month_obs * 100 + 1
last_day_obs = month_obs * 100 + n_days

begin = getDayObsStartTime(first_day_obs)  # first day 12:00 UTC
if INCLUDE_LAST_NIGHT:
    end = getDayObsStartTime(last_day_obs) + 24 * u.hour  # following noon
else:
    end = getDayObsStartTime(last_day_obs) - 1 * u.s  # last day 11:59:59 UTC

print(f"month_obs {month_obs}: {begin.isot} -> {end.isot} (UTC)")

month_obs 202602: 2026-02-01T12:00:00.000 -> 2026-02-28T11:59:59.000 (UTC)


## Query the set-point (re)configuration events

This is a sparse event — a row is written only when a chiller's set point is (re)configured.

In [7]:
df = getEfdData(efd_client := makeEfdClient(), TOPIC, begin=begin, end=end)
print(f"{len(df)} configuration event(s) for month_obs {month_obs}")
df[["device_id", "activeSetpoint"]].head() if len(df) else df

154 configuration event(s) for month_obs 202602


,device_id,activeSetpoint
2026-02-02 01:49:45.759382+00:00,103,0.0
2026-02-02 01:49:58.621641+00:00,103,2.0
2026-02-02 10:12:22.123233+00:00,101,6.5
2026-02-02 10:12:22.140578+00:00,102,5.5
2026-02-03 10:13:23.607063+00:00,101,7.2


## Catastrophic cooling events

Load the curated catalog of glycol / Dynalene / PCS cooling incidents (realised **faults** and averted
**near-misses**) and keep only those inside the plotted window. They are overlaid on the set-point
figure as vertical **dashed** lines — red for realised faults (`outcome == "actual"`), amber for
near-misses — so set-point behaviour can be read against known incidents. Full detail is in the events
table at the bottom.

The catalog spans 2023–2026; several months have no recorded events (e.g. the default `month_obs`
`202601`). Months with 2026 events include `202603` (PCS Chiller2 failure) and `202605` (PCS filter clog).

In [8]:
events = pd.read_csv(EVENTS_CSV)
# Parse the fault onset as UTC, then drop tz for Bokeh's (UTC) datetime axis.
events["t"] = pd.to_datetime(
    events["fault_start_timestamp_utc"], utc=True
).dt.tz_localize(None)

# Keep only events inside the plotted window.
win_lo, win_hi = begin.utc.datetime, end.utc.datetime
events_win = events[(events["t"] >= win_lo) & (events["t"] <= win_hi)].sort_values("t")

print(f"{len(events_win)} catastrophic event(s) within month_obs {month_obs}")
(
    events_win[
        ["event_id", "t", "outcome", "consequence", "camera_system", "fracas_ticket"]
    ]
    if len(events_win)
    else "No catastrophic events recorded in this window."
)

0 catastrophic event(s) within month_obs 202602


'No catastrophic events recorded in this window.'

## Figure — all chillers on a single panel

All four chillers are drawn as **step** lines on one shared panel, one colour each. A set point is
piecewise-constant, so the last value is held to the end of the window. Use the box-zoom /
wheel-zoom / pan tools to inspect clusters of changes.

In [9]:
# Naive UTC datetimes for Bokeh (its datetime axis assumes UTC).
x_lo = begin.utc.datetime
x_hi = end.utc.datetime


def build_figure():
    """Build the single-panel step plot from fresh Bokeh models.

    Bokeh models can belong to only one document, so `save(...)` (file doc) and
    `show(...)` (notebook doc) each need their own set of objects — hence a builder.
    """
    p = figure(
        height=420,
        sizing_mode="stretch_width",
        x_axis_type="datetime",
        x_range=Range1d(start=x_lo, end=x_hi),
        title=f"HVAC chiller active set points — month_obs {month_obs}",
        tools="xpan,xwheel_zoom,box_zoom,reset,save",
        active_scroll="xwheel_zoom",
    )

    for dev_id, (name, color) in CHILLERS.items():
        g = df[df["device_id"] == dev_id].sort_index() if len(df) else df
        if not len(g):
            continue
        # Drop tz (values are UTC) and hold the last value to the end of the window.
        gi = g.index
        gi = gi.tz_localize(None) if gi.tz is not None else gi
        xs = list(gi) + [x_hi]
        ys = list(g["activeSetpoint"]) + [g["activeSetpoint"].iloc[-1]]
        src = ColumnDataSource(dict(x=xs, y=ys))
        step = p.step(
            x="x", y="y", source=src, mode="after",
            line_color=color, line_width=2, legend_label=name,
        )
        p.add_tools(
            HoverTool(
                renderers=[step],
                tooltips=[
                    ("chiller", name),
                    ("time", "@x{%F %T}"),
                    ("set point", "@y{0.00} °C"),
                ],
                formatters={"@x": "datetime"},
                mode="mouse",
            )
        )

    # Overlay catastrophic cooling events as vertical dashed lines (fresh models
    # each call — a Span/Label belongs to a single document, like the glyphs above).
    for _, ev in events_win.iterrows():
        color = EVENT_COLORS.get(ev["outcome"], "#888888")
        p.add_layout(
            Span(
                location=ev["t"],
                dimension="height",
                line_color=color,
                line_dash="dashed",
                line_width=2,
                line_alpha=0.9,
            )
        )
        # Pin a small tag near the top (screen-space y) so it survives auto y-ranging.
        p.add_layout(
            Label(
                x=ev["t"],
                y=8,
                y_units="screen",
                text=f"#{int(ev['event_id'])}",
                text_color=color,
                text_font_size="9pt",
                text_align="center",
                background_fill_color="white",
                background_fill_alpha=0.6,
            )
        )

    p.xaxis.axis_label = "Time [UTC]"
    p.yaxis.axis_label = "Set point [°C]"
    p.legend.location = "top_left"
    p.legend.click_policy = "hide"
    return p


os.makedirs("plots", exist_ok=True)
output_file(f"plots/chiller_setpoints_{month_obs}.html", title=f"Chiller set points {month_obs}")
save(build_figure())  # fresh models -> file document

output_notebook()  # switch rendering back to inline
show(build_figure())  # fresh models -> notebook document

Loading BokehJS ...

## Change-log table

Keep only rows where the set point actually moved (per chiller). This also drops redundant
re-publications (e.g. a CSC restart re-emits the current config unchanged).

In [10]:
if len(df):
    changes = (
        df.sort_index()
        .groupby("device_id")["activeSetpoint"]
        .apply(lambda s: s[s.diff().ne(0)])  # first row + genuine changes only
        .reset_index()
        .rename(columns={"level_1": "time"})
    )
    changes["chiller"] = changes["device_id"].map(lambda d: CHILLERS[d][0])
    changes["delta_C"] = changes.groupby("device_id")["activeSetpoint"].diff()
    display(changes[["time", "chiller", "activeSetpoint", "delta_C"]])
else:
    print("No configuration events for this month_obs.")

,time,chiller,activeSetpoint,delta_C
0,2026-02-02 10:12:22.123233+00:00,chiller01,6.5,NaN
1,2026-02-03 10:13:23.607063+00:00,chiller01,7.2,0.7
2,2026-02-04 10:14:26.590458+00:00,chiller01,8.2,1.0
3,2026-02-06 10:16:25.970467+00:00,chiller01,7.0,-1.2
4,2026-02-07 10:17:33.928491+00:00,chiller01,8.6,1.6
...,...,...,...,...
146,2026-02-27 08:59:32.688504+00:00,chiller03,2.0,2.0
147,2026-02-27 15:06:55.653939+00:00,chiller03,0.0,-2.0
148,2026-02-27 15:07:48.796908+00:00,chiller03,2.0,2.0
149,2026-02-27 15:07:51.132103+00:00,chiller03,0.0,-2.0


## Catastrophic events in this window

Full detail for the events drawn as vertical dashed lines above. The `#N` tags on the plot are the
`event_id` in this table. `summary` and `fracas_url` give the incident write-up and tracking ticket.

In [11]:
if len(events_win):
    with pd.option_context("display.max_colwidth", None):
        display(
            events_win[
                [
                    "event_id",
                    "t",
                    "outcome",
                    "consequence",
                    "camera_system",
                    "fracas_ticket",
                    "fracas_url",
                    "summary",
                ]
            ].set_index("event_id")
        )
else:
    print("No catastrophic events recorded in this window.")

No catastrophic events recorded in this window.
